# 📗 Notebook 2 — Analisi multidimensionale del Liceo Virgilio

Nel **Notebook 1** abbiamo prodotto una *fotografia locale*: una sola classe, un solo quadrimestre, quattro materie linguistiche. È stata utile per cogliere criticità specifiche, ma la dirigenza ora pone una domanda diversa, di livello superiore:

> *"Va bene la 3A, ma le altre sezioni? E rispetto all'anno scorso? Ci sono materie su cui una classe spicca, o sezioni che traggono vantaggio da una particolare combinazione di docenti?"*

In altre parole: serve una **visione d'insieme**, capace di confrontare classi, materie e annualità in un colpo solo.

---

## 📐 La struttura dei dati: un tensore quadridimensionale

Per rispondere a queste domande, organizziamo i dati in un **array NumPy a quattro dimensioni** di forma `(2, 3, 4, 5)`. Ogni asse ha un significato preciso:

| Asse | Dimensione | Significato |
|------|-----------|-------------|
| 0 | 2 | Anni accademici (2023/24, 2024/25) |
| 1 | 3 | Sezioni del terzo anno (3A, 3B, 3C) |
| 2 | 4 | Materie classiche (Latino, Greco, Italiano, Storia) |
| 3 | 5 | Prove svolte nell'anno per ciascuna materia |

Questa struttura permette, con poche operazioni di aggregazione lungo gli assi, di rispondere a domande comparative che con tabelle bidimensionali richiederebbero molti passaggi manuali.

---

## 🧭 Cosa contiene questo notebook

Coerentemente con la natura del problema, **il taglio di questo notebook è più tecnico degli altri due**: lavoriamo prevalentemente su operazioni matriciali di NumPy. Le tre analisi previste sono:

1. **Somma sugli ultimi due assi** — aggregazione del tensore in una matrice (anni × classi) di punteggi cumulati, leggibile come tabella riassuntiva
2. **Prodotto cartesiano** — generazione di scenari di assegnazione per la pianificazione dell'orario
3. **Extra: normalizzazione lungo un asse** — riscalatura dei punteggi per rendere comparabili materie con scale di valutazione diverse

Chiudiamo con una sintesi per la dirigenza e il ponte verso il **Notebook 3**, dove i dati prenderanno la forma tabellare di un report Pandas.

In [4]:
import numpy as np
from itertools import product

# Riproducibilità: fissiamo il seme del generatore casuale.
# Cambiando questo valore si ottiene un dataset diverso ma con la stessa struttura.
np.random.seed(42)

# Tensore voti del Liceo Virgilio
# Forma (2, 3, 4, 5) = (anni, classi, materie, prove)
# Valori interi tra 1 e 9 (NumPy esclude l'estremo superiore in randint)
tensore_voti = np.random.randint(1, 10, size=(2, 3, 4, 5))

# Verifica della struttura del tensore
print("Tensore voti — Liceo Virgilio")
print("-" * 60)
print(f"Numero di dimensioni: {tensore_voti.ndim}")
print(f"Forma:                {tensore_voti.shape}")
print(f"  asse 0 (anni):     {tensore_voti.shape[0]}  → 2023/24, 2024/25")
print(f"  asse 1 (classi):   {tensore_voti.shape[1]}  → 3A, 3B, 3C")
print(f"  asse 2 (materie):  {tensore_voti.shape[2]}  → Latino, Greco, Italiano, Storia")
print(f"  asse 3 (prove):    {tensore_voti.shape[3]}  → cinque prove per materia")
print()
print("Contenuto del tensore:")
print(tensore_voti)

Tensore voti — Liceo Virgilio
------------------------------------------------------------
Numero di dimensioni: 4
Forma:                (2, 3, 4, 5)
  asse 0 (anni):     2  → 2023/24, 2024/25
  asse 1 (classi):   3  → 3A, 3B, 3C
  asse 2 (materie):  4  → Latino, Greco, Italiano, Storia
  asse 3 (prove):    5  → cinque prove per materia

Contenuto del tensore:
[[[[7 4 8 5 7]
   [3 7 8 5 4]
   [8 8 3 6 5]
   [2 8 6 2 5]]

  [[1 6 9 1 3]
   [7 4 9 3 5]
   [3 7 5 9 7]
   [2 4 9 2 9]]

  [[5 2 4 7 8]
   [3 1 4 2 8]
   [4 2 6 6 4]
   [6 2 2 4 8]]]


 [[[7 9 8 5 2]
   [5 8 9 9 1]
   [9 7 9 8 1]
   [8 8 3 1 8]]

  [[3 3 1 5 7]
   [9 7 9 8 2]
   [1 7 7 8 5]
   [3 8 6 3 1]]

  [[3 5 3 1 5]
   [7 7 9 3 7]
   [1 4 4 5 7]
   [7 4 7 3 6]]]]


## 1️⃣ Task 1 — Punteggio cumulato per classe e anno

**Domanda della dirigenza.** *"Qual è il rendimento complessivo di ogni sezione, anno per anno?"*

Per rispondere serve **collassare** due delle quattro dimensioni del tensore: quella delle **materie** (asse 2) e quella delle **prove** (asse 3). Sommando lungo questi due assi otteniamo, per ogni combinazione (anno, classe), un singolo numero che rappresenta il punteggio totale accumulato — la somma di tutti i voti, in tutte le materie, in tutte le prove.

### Perché proprio gli ultimi due assi

NumPy permette di sommare lungo qualsiasi combinazione di assi. La scelta `axis=(-2, -1)` non è arbitraria: stiamo decidendo *cosa tenere* e *cosa aggregare*.

| Asse | Significato | Destino |
|------|-------------|---------|
| 0 (anni) | 2023/24, 2024/25 | **Tenuto** — vogliamo confrontare gli anni |
| 1 (classi) | 3A, 3B, 3C | **Tenuto** — vogliamo confrontare le sezioni |
| 2 (materie) | Latino, Greco, Italiano, Storia | *Aggregato* — somma su tutte le materie |
| 3 (prove) | cinque prove dell'anno | *Aggregato* — somma su tutte le prove |

Il risultato è una **matrice 2 × 3** (anni × classi) — esattamente la tabella che la dirigenza si aspetta di vedere in un consiglio docenti.

In [5]:
# Punteggio cumulato per classe e anno: somma del tensore lungo gli assi
# delle materie (asse 2) e delle prove (asse 3).
# Risultato: matrice di forma (anni, classi) = (2, 3)
punteggi_cumulati = np.sum(tensore_voti, axis=(-2, -1))

# Etichette per la stampa leggibile
anni = ["2023/24", "2024/25"]
classi = ["3A", "3B", "3C"]

# Stampa come tabella formattata
print("Punteggio cumulato per classe e anno — Liceo Virgilio")
print("-" * 60)

# Intestazione di colonna
print(f"{'Anno':<10}", end="")
for classe in classi:
    print(f"{classe:>10}", end="")
print()

# Righe della tabella
for i, anno in enumerate(anni):
    print(f"{anno:<10}", end="")
    for j in range(len(classi)):
        print(f"{punteggi_cumulati[i, j]:>10}", end="")
    print()

print("-" * 60)
print(f"Forma della matrice aggregata: {punteggi_cumulati.shape}")
print(f"Punteggio totale del Liceo (entrambi gli anni): {punteggi_cumulati.sum()}")

# Lettura sintetica
print()
print("Lettura rapida:")
classe_top_per_anno = [classi[j] for j in punteggi_cumulati.argmax(axis=1)]
for i, anno in enumerate(anni):
    totale_anno = punteggi_cumulati[i].sum()
    print(f"  • {anno}: classe migliore {classe_top_per_anno[i]}, totale annuo {totale_anno}")

Punteggio cumulato per classe e anno — Liceo Virgilio
------------------------------------------------------------
Anno              3A        3B        3C
2023/24          111       105        88
2024/25          125       103        98
------------------------------------------------------------
Forma della matrice aggregata: (2, 3)
Punteggio totale del Liceo (entrambi gli anni): 630

Lettura rapida:
  • 2023/24: classe migliore 3A, totale annuo 304
  • 2024/25: classe migliore 3A, totale annuo 326


## 2️⃣ Task 2 — Generazione di scenari di assegnazione

**Domanda di partenza.** *"Quanti modi diversi ci sono di assegnare un insegnante a una materia, in una certa aula, in una certa fascia oraria?"*

Questa domanda è **il primo passo della pianificazione dell'orario**: prima di applicare i vincoli (un docente non può tenere due lezioni in contemporanea, certe materie hanno bisogno di laboratori specifici, ecc.), bisogna sapere qual è l'**universo delle combinazioni possibili** dal quale si parte. Su questo universo si applicheranno poi i vincoli, restringendo lo spazio delle soluzioni ammissibili.

L'operazione matematica che genera questo universo è il **prodotto cartesiano**: dati N insiemi, produce tutte le combinazioni che si possono formare prendendo un elemento da ciascuno. In Python lo strumento canonico è `itertools.product`.

### Cornice del task — onestà intellettuale

Coerentemente con quanto dichiarato nel **Notebook 0**, segnaliamo che questo task ha natura prevalentemente tecnica: il codice dimostra la padronanza di `itertools.product`, mentre la pianificazione d'orario reale richiederebbe poi un solver dedicato (programmazione lineare intera, constraint propagation, o euristiche specifiche) per filtrare lo spazio delle combinazioni in base ai vincoli. Quello che facciamo qui è il **primo passo** di quel workflow, non il workflow completo.

### I quattro vettori di input

Per rendere immediatamente leggibile l'output, scegliamo vettori **semanticamente etichettati** invece che numerici:

| Vettore | Cardinalità | Contenuto |
|---------|-------------|-----------|
| Insegnanti | 3 | Prof. Rossi, Prof. Bianchi, Prof. Verdi |
| Materie | 3 | Latino, Greco, Italiano |
| Fasce orarie | 4 | 1ª, 2ª, 3ª, 4ª ora |
| Aule | 2 | Aula 1, Aula Magna |

Il numero totale di combinazioni sarà il **prodotto delle cardinalità**: 3 × 3 × 4 × 2 = **72 scenari**. Per leggibilità ne stamperemo solo i primi 10.

In [6]:
def prodotto_cartesiano(vettori, max_da_stampare=10):
    """
    Calcola e stampa il prodotto cartesiano di una lista di vettori.

    Args:
        vettori (list): Lista di vettori (liste o tuple) di cui calcolare
                        il prodotto cartesiano.
        max_da_stampare (int): Numero massimo di combinazioni da stampare
                               a video. Il conteggio totale viene comunque
                               mostrato. Default: 10.

    Returns:
        list: Lista di tuple, ciascuna rappresentante una combinazione
              del prodotto cartesiano.
    """
    # itertools.product accetta i vettori come argomenti separati;
    # l'operatore * "spacchetta" la lista in argomenti posizionali.
    risultato = list(product(*vettori))

    # Stampa di riferimento dei vettori di input
    print("Vettori di input:")
    for i, v in enumerate(vettori, start=1):
        print(f"  Vettore {i} ({len(v)} elementi): {v}")

    # Stampa delle prime combinazioni
    print(f"\nPrimi {min(max_da_stampare, len(risultato))} scenari (su {len(risultato)} totali):")
    for combinazione in risultato[:max_da_stampare]:
        print(f"  {combinazione}")

    if len(risultato) > max_da_stampare:
        print(f"  ... e altri {len(risultato) - max_da_stampare} scenari non mostrati.")

    print(f"\nNumero totale di combinazioni: {len(risultato)}")

    return risultato


# Vettori semantici letti dal sistema informatico del Liceo Virgilio
insegnanti = ["Prof. Rossi", "Prof. Bianchi", "Prof. Verdi"]
materie = ["Latino", "Greco", "Italiano"]
fasce_orarie = ["1ª ora", "2ª ora", "3ª ora", "4ª ora"]
aule = ["Aula 1", "Aula Magna"]

vettori_pianificazione = [insegnanti, materie, fasce_orarie, aule]

# Esecuzione: generazione dello spazio degli scenari
print("Generazione scenari di assegnazione — Liceo Virgilio")
print("-" * 60)
scenari = prodotto_cartesiano(vettori_pianificazione)

Generazione scenari di assegnazione — Liceo Virgilio
------------------------------------------------------------
Vettori di input:
  Vettore 1 (3 elementi): ['Prof. Rossi', 'Prof. Bianchi', 'Prof. Verdi']
  Vettore 2 (3 elementi): ['Latino', 'Greco', 'Italiano']
  Vettore 3 (4 elementi): ['1ª ora', '2ª ora', '3ª ora', '4ª ora']
  Vettore 4 (2 elementi): ['Aula 1', 'Aula Magna']

Primi 10 scenari (su 72 totali):
  ('Prof. Rossi', 'Latino', '1ª ora', 'Aula 1')
  ('Prof. Rossi', 'Latino', '1ª ora', 'Aula Magna')
  ('Prof. Rossi', 'Latino', '2ª ora', 'Aula 1')
  ('Prof. Rossi', 'Latino', '2ª ora', 'Aula Magna')
  ('Prof. Rossi', 'Latino', '3ª ora', 'Aula 1')
  ('Prof. Rossi', 'Latino', '3ª ora', 'Aula Magna')
  ('Prof. Rossi', 'Latino', '4ª ora', 'Aula 1')
  ('Prof. Rossi', 'Latino', '4ª ora', 'Aula Magna')
  ('Prof. Rossi', 'Greco', '1ª ora', 'Aula 1')
  ('Prof. Rossi', 'Greco', '1ª ora', 'Aula Magna')
  ... e altri 62 scenari non mostrati.

Numero totale di combinazioni: 72


## ➕ Extra — Normalizzazione per confronti equi tra prove

> *Esercizio aggiuntivo, ideato e risolto durante il corso, qui rivisitato come strumento applicativo per l'analisi del Liceo Virgilio.*

### Il problema dei valori grezzi

Quando confrontiamo voti di prove diverse, ci scontriamo con un fatto noto a chiunque abbia insegnato: **non tutte le prove pesano uguale**. Una verifica di Latino può essere più severa di un'interrogazione di Storia, e un voto di 6 in una può valere più di un 7 nell'altra. Confrontare i numeri grezzi rischia di essere fuorviante.

Una tecnica classica per gestire questa asimmetria è la **normalizzazione lungo un asse**: invece di guardare i valori assoluti, ogni voto viene espresso come **quota relativa** rispetto al totale lungo l'asse scelto. La somma lungo quell'asse, dopo la normalizzazione, fa esattamente 1.

### Cosa normalizziamo

Per non perdere di vista la struttura del nostro tensore, lavoriamo su un **sotto-cubo** — fissiamo l'anno 2023/24 e prendiamo tutto il resto, ottenendo una matrice tridimensionale di forma `(3, 4, 5)` = (classi × materie × prove).

Normalizziamo **lungo l'asse delle prove (asse 2)**. L'interpretazione è la seguente:

> *Per ogni coppia (classe, materia), il valore normalizzato della n-esima prova rappresenta la quota di quella prova nel totale dei voti di quella materia per quella classe nell'anno.*

In altre parole, otteniamo una **traiettoria temporale relativa**: invece di sapere che la 3A ha preso 7, 8, 5, 6, 9 in Latino, sappiamo che ha distribuito il proprio rendimento come 20%, 23%, 14%, 17%, 26% del totale annuo in quella materia.

### Perché può servire alla dirigenza

Questo formato rende immediatamente visibili:
- **Trend di crescita o calo** durante l'anno (se le quote crescono o decrescono nel tempo)
- **Concentrazioni anomale** (una prova che pesa molto più delle altre, indizio di valutazione disomogenea)
- **Confronti tra classi diverse** sulla stessa materia, indipendentemente dalla severità della scala usata

In [7]:
def normalizza_matrice(A: np.ndarray, axis: int) -> np.ndarray:
    """
    Normalizza un array NumPy N-dimensionale lungo un asse specificato.

    Args:
        A (np.ndarray): Array NumPy di dimensione arbitraria.
        axis (int): Asse lungo il quale effettuare la normalizzazione.

    Returns:
        np.ndarray: Array normalizzato lungo l'asse specificato.
                    La somma degli elementi lungo l'asse scelto sarà 1.

    Raises:
        TypeError: Se A non è un array NumPy o axis non è un intero.
        ValueError: Se l'asse specificato non è valido per le dimensioni dell'array.
    """
    # Controlli sugli input
    if not isinstance(A, np.ndarray):
        raise TypeError("L'input A deve essere un array NumPy.")
    if not isinstance(axis, int):
        raise TypeError("L'asse deve essere un numero intero.")
    if axis < -A.ndim or axis >= A.ndim:
        raise ValueError(f"Asse non valido. L'array ha {A.ndim} dimensioni.")

    # Calcolo della somma lungo l'asse, mantenendo le dimensioni per il broadcasting
    somme = np.sum(A, axis=axis, keepdims=True)

    # Sicurezza: se la somma è zero, evitiamo la divisione per zero
    # sostituendola con 1 (il risultato sarà 0, che è coerente)
    somme = np.where(somme == 0, 1, somme)

    # Normalizzazione vera e propria
    A_normalizzata = A / somme

    return A_normalizzata


# Estrazione del sotto-cubo: anno 2023/24, tutte le classi, tutte le materie, tutte le prove
sotto_cubo_2023 = tensore_voti[0]  # shape (3, 4, 5) = (classi, materie, prove)

# Normalizzazione lungo l'asse delle prove (asse 2)
sotto_cubo_normalizzato = normalizza_matrice(sotto_cubo_2023, axis=2)

# Etichette per la stampa interpretabile
classi = ["3A", "3B", "3C"]
materie = ["Latino", "Greco", "Italiano", "Storia"]
prove = ["Prova 1", "Prova 2", "Prova 3", "Prova 4", "Prova 5"]

print("Normalizzazione lungo le prove — Anno 2023/24")
print("-" * 60)
print(f"Forma sotto-cubo originale:    {sotto_cubo_2023.shape}")
print(f"Forma sotto-cubo normalizzato: {sotto_cubo_normalizzato.shape}")

# Esempio leggibile: traiettoria di una specifica classe in una specifica materia
classe_idx, materia_idx = 0, 0  # 3A, Latino
print(f"\nEsempio — {classi[classe_idx]}, {materie[materia_idx]}:")
print(f"  Voti grezzi:       {sotto_cubo_2023[classe_idx, materia_idx]}")
print(f"  Quote relative:    {np.round(sotto_cubo_normalizzato[classe_idx, materia_idx], 3)}")
print(f"  Somma delle quote: {sotto_cubo_normalizzato[classe_idx, materia_idx].sum():.3f}")

# Verifica numerica: la somma lungo l'asse 2 deve fare 1 ovunque
verifica = sotto_cubo_normalizzato.sum(axis=2)
print(f"\nVerifica: somma lungo l'asse delle prove (deve essere ≈ 1 per ogni classe×materia)")
print(f"  Min: {verifica.min():.6f}")
print(f"  Max: {verifica.max():.6f}")
print(f"  Tutte le somme uguali a 1?  {np.allclose(verifica, 1.0)}")

Normalizzazione lungo le prove — Anno 2023/24
------------------------------------------------------------
Forma sotto-cubo originale:    (3, 4, 5)
Forma sotto-cubo normalizzato: (3, 4, 5)

Esempio — 3A, Latino:
  Voti grezzi:       [7 4 8 5 7]
  Quote relative:    [0.226 0.129 0.258 0.161 0.226]
  Somma delle quote: 1.000

Verifica: somma lungo l'asse delle prove (deve essere ≈ 1 per ogni classe×materia)
  Min: 1.000000
  Max: 1.000000
  Tutte le somme uguali a 1?  True


## 📋 Sintesi per la dirigenza

Le tre analisi di questo notebook hanno restituito una **visione d'insieme** del terzo anno del Liceo che il Notebook 1, focalizzato sulla sola 3A, non poteva offrire.

**Punteggi cumulati.** La 3A è la sezione con rendimento più alto in entrambi gli anni. Il Liceo nel suo complesso mostra un **trend di miglioramento** tra 2023/24 (totale 304) e 2024/25 (totale 326). La 3C, pur in crescita, resta la sezione con i punteggi più bassi e meriterebbe un approfondimento dedicato.

**Scenari di assegnazione.** Il prodotto cartesiano sui quattro vettori di pianificazione (insegnanti, materie, fasce orarie, aule) ha generato 72 scenari teorici. È il primo passo di un processo di scheduling più ampio: lo spazio delle combinazioni va poi ristretto applicando i vincoli operativi (sovrapposizioni, requisiti di aula, ecc.). Il task ha qui valore metodologico più che applicativo immediato.

**Normalizzazione lungo le prove.** Trasformare i voti grezzi in quote relative permette di leggere la distribuzione del rendimento durante l'anno, indipendentemente dalla severità della scala assoluta. Una traiettoria temporale, più che un giudizio puntuale.

---

## 🔜 Ponte verso il Notebook 3

Le matrici NumPy sono lo strumento giusto per l'analisi, ma non per la consegna. Quando i risultati devono arrivare al tavolo della dirigenza in forma riutilizzabile — letta in riunione, esportata, condivisa — la struttura matriciale grezza diventa un ostacolo: le etichette non vivono dentro l'array, le statistiche descrittive vanno calcolate a parte.

Nel **📕 Notebook 3** passiamo a **Pandas**. Lavoreremo su due fonti distinte (un elenco manuale della segreteria e un export del sistema informatico), le combineremo, e produrremo il **report tabellare finale** — il deliverable che la dirigenza userà nei propri incontri.